![hslu_logo.png](img/hslu_logo.png)

## Week 5

<hr style="border:1px solid black">


# Excercise: Activation maps live
---
---
This excercise is to illustrate the activation maps of CNN at different pooling stages.

### Import necessary packages

In [2]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
import time


#### Select the model
We use vgg16 due to its yet simple architecture

In [3]:
#download the model and the weights 
from torchvision.models import vgg16, VGG16_Weights

#### Prepare the model
We use the default weights and set up the model. Some models require to switch between `eval()` and `train()` 

Note that the architecture consists of two parts:
1. `(features)`<br>
   The subsequent convolutional and pooling layers
3. `(classifier)`<br>
   The two dense layers for the classification

In [4]:
weights = VGG16_Weights.DEFAULT
vgg16_model = vgg16(weights=weights)
vgg16_model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

### Setup Live Processing

We use the `classify_image` method (notice the parameter `limit_score`) to show the most relevant categories in an image.

In [5]:
def classify_image(image, model=vgg16_model, weights=weights, limit_score=0.1):
    img_trans = weights.transforms(antialias=True)(image)
    prediction = vgg16_model(img_trans.unsqueeze(0)).softmax(1)
 
    high_predictions, class_ids = torch.sort(prediction[0,:], descending=True)
    relevant_indices = torch.where(high_predictions > limit_score)[0]

    out_strings = []
    for ind in relevant_indices:
        out_strings.append(f'score: {int(100*high_predictions[ind])}%: {weights.meta["categories"][class_ids[ind]]}')

    return out_strings

We also show the activation maps after the different pooling layers (`layer`). Max-pooling or average-pooling (sum) (`select_type=0/1`) can be chosen.

In [6]:
def activation_map(image, model=vgg16_model, weights=weights, layer=30, select_type=0):
    img_trans = weights.transforms(antialias=True)(image)
    
    #determine the feature maps (add +1 to include pooling layer)
    feature_map = vgg16_model.features[:layer+1](img_trans).detach().numpy()

    if select_type == 0:
        out_map = np.max(feature_map, axis=0)
    elif select_type == 1:
        out_map = np.mean(feature_map, axis=0)

    out_map = 255*(out_map - out_map.min()) / (out_map.max() - out_map.min())
    
    return out_map

Simple way to create an upscaled tile-image

In [7]:
def up_scale(image, n_rep):
    #there may be a quicker way
    image_rep = np.repeat(np.repeat(image, n_rep, axis=0), n_rep, axis=1)

    return image_rep

In [ ]:
#get indices of the pooling layers to choose below
pool_index = []
for index, layer in enumerate(vgg16_model.features):
    if layer.__class__.__name__=='MaxPool2d':
        pool_index.append(index)

#bootstrap for pooling layer selection
layer_ind = len(pool_index)-1
layer = pool_index[layer_ind]

select_type = 0
pool_type = 'max pooling'

# Open the default camera
#to check available capture devices best is matlab :-/
capture_device = 1

cam = cv2.VideoCapture(capture_device)

# Get the default frame width and height
frame_width = int(cam.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cam.get(cv2.CAP_PROP_FRAME_HEIGHT))

#center crop image with 448x448 (it seems that vgg center crops image)
x_low = frame_width // 2 - 224
x_up = frame_width // 2 + 224
y_low = frame_height // 2 - 224
y_up = frame_height // 2 + 224


# font
font = cv2.FONT_HERSHEY_SIMPLEX
# fontScale
fontScale = .6
# Blue color in BGR
color = (255, 0, 0)

# Line thickness of 2 px
thickness = 1

while True:
    
    ret, frame = cam.read()

    #center crop
    frame = frame[y_low:y_up, x_low:x_up]

    #image conversion 
    image = torch.zeros((3,448,448),dtype=torch.uint8)
    for plane in range(3):
        image[plane] = torch.tensor(frame[:,:,2-plane])

    start=time.time()
    out_strings = classify_image(image, limit_score=0.1)
    stop=time.time()
    activ_map = activation_map(image, model=vgg16_model, weights=weights, 
                               layer=layer, select_type=select_type)

    #upscale the activation map
    activ_map_up = up_scale(activ_map, 2**(layer_ind+2))
    #activ_map_up = cv2.resize(activ_map, (frame_width, frame_height), 0, 0, cv2.INTER_NEAREST_EXACT)
    
    

    out_strings.append(f'processing time {stop-start:.2f}')
    out_strings.append(f'pooling layer {layer}, tpye is {pool_type}')

    for ind, str in enumerate(out_strings):
        pos = (20, 20*(ind+1))
        cv2.putText(frame, str, pos, font, 
                   fontScale, color, thickness, cv2.LINE_AA)

    # Display the captured frame
    cv2.imshow('Camera', frame)
    cv2.imshow('Activation', activ_map_up.astype(np.uint8))

    key = cv2.waitKeyEx(1)
    if key != -1:
        if key == 101: #'e'
            break
        elif key == 2424832: #left
            layer_ind = max(0, layer_ind-1)
        elif key == 2555904: #right
            layer_ind = min(len(pool_index)-1, layer_ind+1)
        elif key == 2490368: #up
            select_type = 0
            pool_type = 'max pooling'
        elif key == 2621440: #down
            select_type = 1
            pool_type = 'average pooling'
        
        layer = pool_index[layer_ind]

# Release the capture and writer objects
cam.release()
cv2.destroyWindow('Camera')
cv2.destroyWindow('Activation')

KeyboardInterrupt: 